In [2]:
import socket 
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM) 
type(sock)

socket.socket

In [6]:
help(socket.socket.accept)
help(socket.socket)

Help on function accept in module socket:

accept(self)
    accept() -> (socket object, address info)

    Wait for an incoming connection.  Return a new socket
    representing the connection, and the address of the client.
    For IP sockets, the address info is a pair (hostaddr, port).

Help on class socket in module socket:

class socket(_socket.socket)
 |  socket(family=-1, type=-1, proto=-1, fileno=None)
 |
 |  A subclass of _socket.socket adding the makefile() method.
 |
 |  Method resolution order:
 |      socket
 |      _socket.socket
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __enter__(self)
 |
 |  __exit__(self, *args)
 |
 |  __getstate__(self)
 |      Helper for pickle.
 |
 |  __init__(self, family=-1, type=-1, proto=-1, fileno=None)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __repr__(self)
 |      Wrap __repr__() to reveal the real class name and socket
 |      address(es).
 |
 |  accept(self)
 |      accept() -> (socket 

In [ ]:

import socket
import json
import time

UDP_PORT = 3133
BROADCAST = "255.255.255.255"
INTERVAL = 5


def broadcast(sock, peer_id, name, tcp_port):
    packet = {
        "app": "LocalPeer",
        "id": peer_id,
        "name": name,
        "port": tcp_port
    }

    data = json.dumps(packet).encode("utf-8")
    sock.sendto(data, (BROADCAST, UDP_PORT))


def discovery_loop(sock, stop, peer_id, name, tcp_port, peers, lock):
    next_scan = 0

    while not stop.is_set():

        if time.monotonic() >= next_scan:
            try:
                broadcast(sock, peer_id, name, tcp_port)
            except OSError as error:
                print("Broadcast error:", error)

            next_scan = time.monotonic() + INTERVAL

        try:
            data, address = sock.recvfrom(4096)
        except socket.timeout:
            continue
        except OSError as error:
            print("Discovery error:", error)
            break

        try:
            packet = json.loads(data.decode("utf-8"))
        except (ValueError, RecursionError):
            continue

        if not isinstance(packet, dict) or packet.get("app") != "LocalPeer":
            continue

        other_id = packet.get("id")
        other_name = packet.get("name")
        port = packet.get("port")

        if not isinstance(other_id, str) or other_id == peer_id:
            continue

        if not isinstance(other_name, str) or not 1 <= len(other_name) <= 32:
            continue

        if not other_name.isprintable() or type(port) is not int or not 1 <= port <= 65535:
            continue

        lock.acquire()
        try:
            peers[other_id] = (other_name, address[0], port)
        finally:
            lock.release()



In [7]:
dir(socket)

['AF_ALG',
 'AF_APPLETALK',
 'AF_ASH',
 'AF_ATMPVC',
 'AF_ATMSVC',
 'AF_AX25',
 'AF_BLUETOOTH',
 'AF_BRIDGE',
 'AF_CAN',
 'AF_DECnet',
 'AF_ECONET',
 'AF_INET',
 'AF_INET6',
 'AF_IPX',
 'AF_IRDA',
 'AF_KEY',
 'AF_LLC',
 'AF_NETBEUI',
 'AF_NETLINK',
 'AF_NETROM',
 'AF_PACKET',
 'AF_PPPOX',
 'AF_QIPCRTR',
 'AF_RDS',
 'AF_ROSE',
 'AF_ROUTE',
 'AF_SECURITY',
 'AF_SNA',
 'AF_TIPC',
 'AF_UNIX',
 'AF_UNSPEC',
 'AF_VSOCK',
 'AF_WANPIPE',
 'AF_X25',
 'AI_ADDRCONFIG',
 'AI_ALL',
 'AI_CANONNAME',
 'AI_NUMERICHOST',
 'AI_NUMERICSERV',
 'AI_PASSIVE',
 'AI_V4MAPPED',
 'ALG_OP_DECRYPT',
 'ALG_OP_ENCRYPT',
 'ALG_OP_SIGN',
 'ALG_OP_VERIFY',
 'ALG_SET_AEAD_ASSOCLEN',
 'ALG_SET_AEAD_AUTHSIZE',
 'ALG_SET_IV',
 'ALG_SET_KEY',
 'ALG_SET_OP',
 'ALG_SET_PUBKEY',
 'AddressFamily',
 'AddressInfo',
 'BDADDR_ANY',
 'BDADDR_BREDR',
 'BDADDR_LE_PUBLIC',
 'BDADDR_LE_RANDOM',
 'BDADDR_LOCAL',
 'BTPROTO_HCI',
 'BTPROTO_L2CAP',
 'BTPROTO_RFCOMM',
 'BTPROTO_SCO',
 'BT_CHANNEL_POLICY',
 'BT_CHANNEL_POLICY_BREDR_ONLY',
 '